In [1]:
class Individual:
    def __init__(self,genome,fitness = 0,validated=False):
        self.genome = genome
        self.fitness = fitness
        self.validated = validated

In [2]:
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
from sklearn.naive_bayes import MultinomialNB
naive_bayes_mult = MultinomialNB()
from sklearn.metrics import r2_score

In [4]:
path = "data_base/bbc_text_cls_adapted.csv"

In [5]:
data = pd.read_csv(filepath_or_buffer = path)

In [6]:
labels = data["labels"].unique()

In [7]:
X = data["text"]
y = data["labels"]
labels = y.unique()

In [8]:
if y.dtype == object or str:
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y = le.fit_transform(y)
    y = pd.DataFrame(y)

In [9]:
from sklearn.model_selection import train_test_split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.5, random_state = 1, stratify = y)

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english')

X_train_transformed = vectorizer.fit_transform(X_train)
X_test_transformed = vectorizer.transform(X_test)

In [12]:
indices_treino = y_train.index
columns = data.iloc[indices_treino, 2:]

In [13]:
columns = columns.apply(le.transform)

In [14]:
matrix = np.array(columns)

In [15]:
matrix

array([[4, 1, 1, ..., 4, 3, 0],
       [3, 3, 3, ..., 0, 4, 4],
       [3, 3, 3, ..., 3, 3, 3],
       ...,
       [0, 0, 0, ..., 3, 4, 4],
       [3, 3, 2, ..., 0, 4, 4],
       [1, 1, 1, ..., 0, 3, 3]], shape=(1112, 20))

In [16]:
individual_lenght = (len(data.columns)-2)

In [17]:
individual_lenght

20

In [18]:
labels = le.transform(labels)

In [19]:
def genome_generator(individual_lenght):
    genome = np.array([])

    for i in range(individual_lenght):
        genome = np.append(genome,random.random())
    return(genome)

In [20]:
def create_initial_population(size,individual_lenght,labels,matrix,X_train_transformed):
    population = []
    for _ in range(size):
        genome = genome_generator(individual_lenght)
        individual = Individual(genome=genome)
        fitness = fitness_function(individual, matrix, X_train_transformed)
        individual.fitness = fitness
        population.append(individual)
        
    return population

In [21]:
def determinator(tiny_list,individual):
    choice = random.choices(tiny_list, weights=individual, k=1)[0]
    return(choice)

In [22]:
def fitness_function(individual,matrix,X_train_transformed):
    genome = individual.genome
    choices = np.array([])
    choices = np.apply_along_axis(lambda x: determinator(x, genome), axis=1, arr=matrix)

    naive_bayes_mult.fit(X_train_transformed,choices)
    naive_bayes_mult.predict(X_train_transformed)
    df_ = naive_bayes_mult.predict_log_proba(X_train_transformed)
    list_before_sum = [max(i) for i in df_]
    fitness = sum(list_before_sum)
    
    return(fitness)

In [23]:
def make_list_proba(df_):
    total_proba_ = df_[0].prod()
    total_proba_2 = df_[1].prod()
    total_proba_3 = df_[2].prod()
    total_proba_4 = df_[3].prod()
    total_proba_5 = df_[4].prod()
    list_proba_ = []
    list_proba_.append(total_proba_)
    list_proba_.append(total_proba_2)
    list_proba_.append(total_proba_3)
    list_proba_.append(total_proba_4)
    list_proba_.append(total_proba_5)
    return(list_proba) 

In [24]:
def selection(selected_tournment):
    selected = []
    for _ in range(len(selected_tournment)):
        first_pair = [selected_tournment[0],selected_tournment[1]]
        second_pair = [selected_tournment[2],selected_tournment[3]]
        winner_1 = max(first_pair, key=lambda x: x.fitness)
        winner_2 = max(second_pair, key=lambda x: x.fitness)
        selected.append(winner_1)
        selected.append(winner_2)
    return selected

In [25]:
def genome_decision(genome_1,genome_2):
    genome_1 = list(genome_1)
    genome_2 = list(genome_2)

    len_ = len(genome_1)
    
    decision_parent = []
    genome_child = []
    
    for ind in range(len_):
        decision_parent.append(random.randint(0,1))

    for ind in range(len_):
        if decision_parent[ind] == 1:
            genome_child.append(genome_1[ind])
        else:
            genome_child.append(genome_2[ind])

    return(genome_child)

In [26]:
def crossover(selected_1, selected_2,matrix,X_train_transformed):
    
    child_1 = Individual(genome = genome_decision(selected_1.genome,selected_2.genome))
    child_2 = Individual(genome = genome_decision(selected_2.genome,selected_1.genome))
    
    fitness_1 = fitness_function(child_1, matrix, X_train_transformed)
    fitness_2 = fitness_function(child_2, matrix, X_train_transformed)

    child_1.fitness = fitness_1
    child_2.fitness = fitness_2
    
    return child_1, child_2

In [27]:
def mutation(child, mutation_rate,labels,matrix,X_train_transformed):
    genome = child.genome
    child.validated = False
    
    for i in range(len(genome)):
        if random.random() < mutation_rate:
            sorted_index = random.randrange(len(genome))
            if(genome[sorted_index]==1):
                genome[sorted_index] = random.choice(labels)
                
    child.genome = genome
    child.fitness = fitness_function(child,matrix,X_train_transformed)
    
    return(child)

In [28]:
def select_individuals(population):
    available = range(len(population))
    sorted_parents = random.sample(available,4)

    parent1 = population[sorted_parents[0]]
    parent2 = population[sorted_parents[1]]
    parent3 = population[sorted_parents[2]]
    parent4 = population[sorted_parents[3]]

    selected_tournment = [parent1,parent2,parent3,parent4]
    
    selected_crossover = selection(selected_tournment)

    return(selected_crossover)

In [29]:
# Main genetic algorithm function
def genetic_algorithm(population_size,generations, mutation_rate,labels,individual_lenght,matrix,X_train_transformed):
    population = create_initial_population(population_size,individual_lenght,labels,matrix,X_train_transformed)

    for generation in range(generations):
        new_population = []

        best_individual = max(population, key=lambda x: fitness_function(x, matrix, X_train_transformed))
        best_fitness = fitness_function(best_individual, matrix, X_train_transformed)

        index_best_individuals = sorted(range(len(population)), key=lambda x: population[x].fitness)
        primary_best_individual = population[index_best_individuals[0]]
        second_best_individual = population[index_best_individuals[1]]

        for _ in range(int((population_size-2)/2)):

            selected_crossover = select_individuals(population)
    
            child_1, child_2 = crossover(selected_crossover[0], selected_crossover[1], matrix, X_train_transformed)
    
            child_1 = mutation(child_1, mutation_rate, labels, matrix, X_train_transformed)
            child_2 = mutation(child_2, mutation_rate, labels, matrix, X_train_transformed)
            
            # sorted_index = random.sample(range(len(population)),2)
            # print(sorted_index)
            # population[sorted_index[0]] = child1
            # population[sorted_index[1]] = child2
            
            # index_worst_individuals = sorted(range(len(population)), key=lambda x: population[x].value)
            new_population.append(child_1)
            new_population.append(child_2)

        new_population.append(primary_best_individual)
        new_population.append(second_best_individual)

        population = new_population

    return (best_individual)

In [30]:
best_individual = genetic_algorithm(20,50,1,labels,individual_lenght,matrix,X_train_transformed)

In [31]:
best_individual.fitness

np.float64(-810.7405105447306)

In [32]:
best_individual.genome

[np.float64(0.9548918451119294),
 np.float64(0.2574023282359629),
 np.float64(0.4470000726353478),
 np.float64(0.9576599238091176),
 np.float64(0.6995731108017607),
 np.float64(0.6012177774711353),
 np.float64(0.8820955252815471),
 np.float64(0.7465721714640986),
 np.float64(0.42812472586697736),
 np.float64(0.6096950033488936),
 np.float64(0.12038579218868228),
 np.float64(0.05968497097307279),
 np.float64(0.2773254051068018),
 np.float64(0.5618565736551451),
 np.float64(0.7014976592263279),
 np.float64(0.9554839364185236),
 np.float64(0.0912128781182846),
 np.float64(0.4091882482492174),
 np.float64(0.01824629917246723),
 np.float64(0.14966630081149102)]